In [0]:
import yaml

print("PyYAML is available!")

In [0]:
from src.common.config_reader import load_config

config = load_config("songs_pipeline")

print(config)

In [0]:
from src.common.config_reader import load_config

config = load_config("songs_pipeline")

print(config["reader_options"])
print(config["bronze_write_mode"])

In [0]:
from src.pipelines.bronze import run_bronze

print("Bronze module imported successfully!")

In [0]:
from src.common.config_reader import load_config
from src.pipelines.bronze import run_bronze

# Load pipeline configuration
config = load_config("songs_pipeline")

# Execute Bronze processing
run_bronze(spark, config)

In [0]:
%sql
SELECT *
FROM workspace.default.songs_raw_table
LIMIT 10;

In [0]:
source_path = "/databricks-datasets/songs/data-001/"

files = dbutils.fs.ls(source_path)

for file in files:
    print(file.path, file.size)

In [0]:
raw_files = [
    f.path for f in files
    if not f.isDir()
]

print("Files found:", len(raw_files))

for path in raw_files[:3]:
    print(f"\nReading: {path}")
    print(dbutils.fs.head(path, 1000))

In [0]:
print("HEADER FILE:")
print(
    dbutils.fs.head(
        "dbfs:/databricks-datasets/songs/data-001/header.txt",
        2000
    )
)

In [0]:
print("DATA FILE:")
print(
    dbutils.fs.head(
        "dbfs:/databricks-datasets/songs/data-001/part-00000",
        2000
    )
)

In [0]:
source_path = "/databricks-datasets/songs/data-001/part-*"

df_raw = (
    spark.read
    .format("csv")
    .option("header", "false")
    .option("delimiter", "\t")
    .option("inferSchema", "true")
    .load(source_path)
)

print("Number of columns:", len(df_raw.columns))

df_raw.printSchema()

display(df_raw.limit(5))

In [0]:
header_content = dbutils.fs.head(
    "dbfs:/databricks-datasets/songs/data-001/header.txt",
    5000
)

print(header_content)

In [0]:
from src.common.config_reader import load_config
from src.pipelines.bronze import build_spark_schema

# Load YAML configuration
config = load_config("songs_pipeline")

# Build Spark schema
source_schema = build_spark_schema(config["schema"])

# Read source data only
df_check = (
    spark.read
    .format(config["source_format"])
    .schema(source_schema)
    .options(**config["reader_options"])
    .load(config["source_path"])
)

# Validate
print("Number of columns:", len(df_check.columns))

df_check.printSchema()

display(df_check.limit(5))

In [0]:
print("Total records:", df_check.count())

print(
    "Records with null song_id:",
    df_check.filter("song_id IS NULL").count()
)

print(
    "Records with null partial_sequence:",
    df_check.filter("partial_sequence IS NULL").count()
)

print("Schema validation completed!")

In [0]:
%sql
SELECT *
FROM workspace.default.songs_raw_table
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.default.songs_raw_table; --31389

In [0]:
%sql
DESCRIBE TABLE workspace.default.songs_raw_table;

In [0]:
from src.common.config_reader import load_config
from src.pipelines.bronze import run_bronze

# Load pipeline configuration
config = load_config("songs_pipeline")

# Execute Bronze ingestion
run_bronze(spark, config)

In [0]:
import importlib
import src.pipelines.bronze as bronze

# Reload the updated Python module
importlib.reload(bronze)

# Load YAML configuration
from src.common.config_reader import load_config
config = load_config("songs_pipeline")

# Build schema
schema = bronze.build_spark_schema(config["schema"])

print("Number of schema fields:", len(schema.fields))
print(schema)

In [0]:
from src.pipelines.bronze import run_bronze

run_bronze(spark, config)

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.default.songs_raw_table;

In [0]:
%sql
DESCRIBE TABLE workspace.default.songs_raw_table;

In [0]:
%sql
SELECT
    artist_id,
    artist_name,
    duration,
    song_id,
    title,
    year,
    ingested_at
FROM workspace.default.songs_raw_table
LIMIT 10;


In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT song_id) AS unique_song_ids,
    COUNT(*) - COUNT(DISTINCT song_id) AS duplicate_count
FROM workspace.default.songs_raw_table;

In [0]:
%sql
SELECT
    song_id,
    COUNT(*) AS occurrence_count
FROM workspace.default.songs_raw_table
GROUP BY song_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

In [0]:
%sql
SELECT
    song_id,
    artist_id,
    artist_name,
    title,
    release,
    duration,
    year,
    tempo,
    ingested_at
FROM workspace.default.songs_raw_table
WHERE song_id IN (
    'SOEGXYE12AF729D9FF',
    'SOCXCVQ12AB017A92A'
)
ORDER BY song_id;

In [0]:
%sql
SELECT *
FROM workspace.default.songs_raw_table
WHERE song_id IN (
    'SOCXCVQ12AB017A92A',
    'SOEGXYE12AF729D9FF'
)
ORDER BY song_id;

In [0]:
from src.pipelines.silver import clean_silver_data

# Read Bronze table
df_bronze = spark.read.table(config["bronze_table"])

# Apply Silver transformations
df_silver_test = clean_silver_data(df_bronze)

# Display selected columns
display(
    df_silver_test.select(
        "artist_name",
        "title",
        "song_title"
    ).limit(10)
)

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df_silver_test.count())

print("Bronze columns:", len(df_bronze.columns))
print("Silver columns:", len(df_silver_test.columns))

In [0]:
from pyspark.sql.functions import col, trim, when, sum

df_silver_test.select(
    sum(
        when(col("artist_name").isNull(), 1).otherwise(0)
    ).alias("null_artist_names"),

    sum(
        when(trim(col("artist_name")) == "", 1).otherwise(0)
    ).alias("blank_artist_names"),

    sum(
        when(col("song_title").isNull(), 1).otherwise(0)
    ).alias("null_song_titles"),

    sum(
        when(trim(col("song_title")) == "", 1).otherwise(0)
    ).alias("blank_song_titles")
).show()


In [0]:
from pyspark.sql.functions import col

display(
    df_silver_test
    .filter(col("song_title").isNull())
    .select(
        "song_id",
        "artist_id",
        "artist_name",
        "title",
        "song_title",
        "release",
        "year",
        "duration"
    )
)


In [0]:
from pyspark.sql.functions import col, sum, when

df_silver_test.select(
    sum(
        when(col("year") == 0, 1).otherwise(0)
    ).alias("zero_year_records"),

    sum(
        when(col("year").isNull(), 1).otherwise(0)
    ).alias("null_year_records")
).show()


In [0]:
%sql
SELECT
    year,
    COUNT(*) AS record_count
FROM workspace.default.songs_raw_table
GROUP BY year
ORDER BY year;

In [0]:
import importlib
import src.pipelines.silver as silver

importlib.reload(silver)

df_silver_test = silver.clean_silver_data(df_bronze)

In [0]:
from pyspark.sql.functions import sum, when

df_silver_test.select(
    sum(
        when(col("year").isNull(), 1).otherwise(0)
    ).alias("null_year_records"),

    sum(
        when(col("year") == 0, 1).otherwise(0)
    ).alias("zero_year_records")
).show()

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df_silver_test.count())

print("Silver columns:", df_silver_test.columns)

In [0]:
import importlib
import src.pipelines.silver as silver

importlib.reload(silver)

from src.common.config_reader import load_config

config = load_config("songs_pipeline")

silver.run_silver(spark, config)

In [0]:
df_silver = spark.read.table(config["silver_table"])

print("Silver record count:", df_silver.count())
print("Silver column count:", len(df_silver.columns))
print("Silver columns:", df_silver.columns)

In [0]:
df_silver.select(
    sum(when(col("year").isNull(), 1).otherwise(0))
    .alias("null_year_records"),

    sum(when(col("year") == 0, 1).otherwise(0))
    .alias("zero_year_records")
).show()

In [0]:
from src.common.config_reader import load_config

config = load_config("songs_pipeline")

print(config["silver_table"])

In [0]:
import importlib
import src.pipelines.gold as gold

importlib.reload(gold)

# Read existing Silver table
df_silver = spark.read.table(config["silver_table"])

# Apply Gold transformations
df_gold_test = gold.build_gold_data(df_silver)

# Display results
display(df_gold_test.limit(20))

In [0]:
df_gold_test.groupBy("rank") \
    .count() \
    .orderBy("rank") \
    .show()

In [0]:
df_gold_test.filter(
    (df_gold_test["rank"].isNull()) |
    (df_gold_test["rank"] > 5) |
    (df_gold_test["year"].isNull())
).count()

In [0]:
import importlib
import src.pipelines.gold as gold

importlib.reload(gold)

from src.common.config_reader import load_config

config = load_config("songs_pipeline")

gold.run_gold(spark, config)

In [0]:
df_gold = spark.read.table(config["gold_table"])

print("Gold record count:", df_gold.count())

print("Gold table schema:")
df_gold.printSchema()

display(df_gold.limit(20))

In [0]:
from src.common.logger import get_logger

logger = get_logger("songs_pipeline")

logger.info("Pipeline started")
logger.warning("This is a warning message")
logger.error("This is an error message")

In [0]:
from src.common.exceptions import (
    PipelineException,
    ConfigurationException,
    DataIngestionException
)

try:
    raise DataIngestionException(
        "Unable to read songs source data"
    )

except PipelineException as e:
    print(f"Pipeline error: {e}")

In [0]:
import importlib
import src.pipelines.bronze as bronze

importlib.reload(bronze)

from src.common.config_reader import load_config

config = load_config("songs_pipeline")

bronze.run_bronze(spark, config)

In [0]:
df_bronze = spark.read.table(config["bronze_table"])

print("Bronze record count:", df_bronze.count())
print("Bronze column count:", len(df_bronze.columns))

In [0]:
import importlib
import src.pipelines.silver as silver

importlib.reload(silver)

silver.run_silver(spark, config)

In [0]:
df_silver = spark.read.table(config["silver_table"])

print("Silver record count:", df_silver.count())
print("Silver column count:", len(df_silver.columns))

In [0]:
import importlib
import src.pipelines.gold as gold

importlib.reload(gold)

gold.run_gold(spark, config)

In [0]:
df_gold = spark.read.table(config["gold_table"])

print("Gold record count:", df_gold.count())
print("Gold column count:", len(df_gold.columns))